In [1]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
import re
from sklearn.decomposition import PCA
from functools import reduce
import gc
from IPython.display import display


## Setup
Run the notebook from the Econ681 folder. Each cohort has a separate sample table showing
employment rows and respondents remaining after the final cleaning restrictions.
Instrument variables are constructed in their own cells before the CSV exports.


In [2]:
# Defaults support Run All from the Econ681 folder; overrides allow a staged run.
PROJECT = Path(os.environ.get('NLSY_PROJECT', str(Path.cwd()))).resolve()
CLEANED_DIR = Path(os.environ.get('NLSY_CLEANED_DIR', str(PROJECT / 'cleaned'))).resolve()
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
data_folder = PROJECT / 'extracted_data'
survey_year_79 = [y for y in range(1979,2023) if y not in [1995, 1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021]]
survey_year_97 = [y for y in range(1997,2024) if y not in [2012,2014,2016,2018,2020,2022]]
race_mapping = {
    3: 'White', 5: 'White', 6: 'White', 7: 'White', 11: 'White', 12: 'White', 
    22: 'White', 23: 'White', 24: 'White', 25: 'White', 27: 'White', 29: 'White', 
    1: 'Black',
    15: 'Hispanic', 16: 'Hispanic', 17: 'Hispanic', 18: 'Hispanic', 19: 'Hispanic', 
    20: 'Hispanic', 21: 'Hispanic',
    2: 'Asian/PI', 4: 'Asian/PI', 8: 'Asian/PI', 10: 'Asian/PI', 13: 'Asian/PI', 
    14: 'Asian/PI', 26: 'Asian/PI',
    0: 'Native/Other', 9: 'Native/Other', 28: 'Native/Other' 
}
sex_mapping = {1:"Male", 2:"Female"}
race_map_97 = {1: 'Black', 2:'Hispanic', 3:'Mixed Race', 4:'Non-black/Non-Hispanic'}


In [3]:
def create_rnum_dict(file_path):
    rnum_to_var = {}
    pattern = re.compile(r"^([A-Za-z0-9]+\.\d{2}[A-Za-z0-9]*)\s+\[(.*?)\]\s+Survey Year:\s*(.*)")
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            match = pattern.search(line.strip())
            if match:
                rnum = re.sub(r'[^a-zA-Z0-9]', '', match.group(1))
                var_name = match.group(2)
                survey_year_79 = match.group(3).strip()
                if re.match(r"^\d{4}$", survey_year_79):
                    var_name = f"{var_name}_{survey_year_79}"
                rnum_to_var[rnum] = var_name     
    return rnum_to_var

# 79 rename dictionary
def rename_columns(col):
    if 'EMPLOYERS_ALL_' in col:
        clean_name = col.replace('EMPLOYERS_ALL_', '')
        if 'UID' in col:
            return clean_name
        match_gap = re.match(r'(.+)_(\d{4})\.(\d+)\.(\d+)$',clean_name)
        if match_gap:
            var, year, job_num, gap_num = match_gap.groups()
            return f"{var}.{job_num}_{gap_num}_{year}"
        match_job = re.match(r'(.+)_(\d{4})\.(\d+)$',clean_name)
        if match_job:
            var, year, job_num = match_job.groups()
            return f"{var}.{job_num}_{year}"
        return clean_name
    elif 'CASEID' in col:
        return 'CASEID'
    elif 'HGCREV' in col:
        parts = col.split('_')
        return f"{parts[0][:-2]}_{parts[1]}"
    elif 'FAM-30_1_1979' in col:
        return 'race'
    elif 'SAMPLE_SEX_1979' in col:
        return 'sex'
    return col


## Occupation mappings


In [4]:
occ70_dict = {}
for x in range(0, 196):
    occ70_dict[x] = "Professional"
for x in range(201, 246):
    occ70_dict[x] = "Managers"
for x in range(260,281):
    occ70_dict[x] = "Sales"
for x in range(301,396):
    occ70_dict[x] = "Clerical"
for x in range(401,581):
    occ70_dict[x] = "Craftsmen"
for x in range(601,786):
    occ70_dict[x] = "Operatives and Laborers"
for x in range(901,985):
    occ70_dict[x] = "Service"

    
occ00_dict = {}
for x in [range(62,71),80,82,93,range(100,352),353,354,361,362,363,462,493,601,903,904]:
    if type(x) is int:
        occ00_dict[x] = "Professional"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Professional"
for x in [range(0,54),56,72,73,83,range(84,92),94,95,931]:
    if type(x) is int:
        occ00_dict[x] = "Managers"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Managers"
for x in [range(475,493),range(494,497)]:
    if type(x) is int:
        occ00_dict[x] = "Sales"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Sales"
for x in [54,60,81,472,474,500,range(510,556),range(556,594)]:
    if type(x) is int:
        occ00_dict[x] = "Clerical"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Clerical"
for x in [352,441,range(621,626),range(630,637),range(642,654),range(666,672),674,676,range(700,761),range(762,771),774,780,803,806,813,816,range(823,826),
          833,835,844,845,range(850,853),range(860,864),875,876,891,900,920,941,range(951,957)]:
    if type(x) is int:
        occ00_dict[x] = "Craftsmen"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Craftsmen"
for x in [range(501,504),640,range(680,692),694,range(771,774),775,range(781,803),804,range(810,813),814,815,range(820,823),range(826,833),834,range(836,843),846,range(853,856),
          range(864,875),range(880,891),range(892,895),896,range(911,916),923,933,935,936,942,950,960,964,965,973,421,425,434,435,600,range(602,614),626,660,672,673,675,692
          ,693,761,895,range(961,964),972,974,975]:
    if type(x) is int:
        occ00_dict[x] = "Operatives and Laborers"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Operatives and Laborers"
for x in [360,364,365,range(370,421),range(422,425),430,432,range(442,462),464,465,934]:
    if type(x) is int:
        occ00_dict[x] = "Service"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Service"


In [5]:
OCC_CODES = {
    'Professional': 1, 'Managers': 2, 'Sales': 3, 'Clerical': 4,
    'Craftsmen': 5, 'Operatives and Laborers': 6, 'Service': 7,
}


## Inputs and price adjustment


In [6]:
# Reuse the existing extracted inputs. zip_extract.py clears its output folders,
# so extraction is deliberately separate from routine cleaning runs.
required_stems = ['79-experience', '79-job_gap', 'nlsy79', '97-tenure',
                  'nlsy97', '97-emp_weekly_array']
missing_inputs = [str(data_folder / (name + ext))
                  for name in required_stems for ext in ('.csv', '.cdb')
                  if not (data_folder / (name + ext)).is_file()]
if missing_inputs:
    raise FileNotFoundError('Extract the required source files first: ' + ', '.join(missing_inputs))


In [7]:
# Convert wages to 2023 dollars
df_cpi = pd.read_csv(PROJECT / 'cpi.csv')
df_cpi['observation_date'] = df_cpi['observation_date'].str[:4].astype(int)
base = df_cpi.loc[df_cpi['observation_date'] == 2023, 'CPIAUCSL'].values[0]
df_cpi['ratio'] = df_cpi['CPIAUCSL'] / base
cpi_map = df_cpi.set_index('observation_date')['ratio'].to_dict()


## NLSY79 employment durations


In [8]:
df79_exp = pd.read_csv(os.path.join(data_folder, '79-experience.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '79-experience.cdb'))
df79_exp = df79_exp.rename(columns = rename_dict)
df79_exp = df79_exp.rename(columns = rename_columns)
df79_exp = df79_exp.replace([-1, -2, -3, -4,-5], np.nan)

# Start Week
start_cols = [col for col in df79_exp.columns if col.startswith("STARTWEEK.")]
df79_startweek_long = df79_exp[["CASEID"] + start_cols].melt(id_vars="CASEID",value_vars=start_cols,var_name="variable",value_name="Startweek")
df79_startweek_long = df79_startweek_long.dropna(subset=['Startweek'])
df79_startweek_long[["job_num","year"]] = df79_startweek_long["variable"].str.extract(r"STARTWEEK\.(\d+)_(\d{4})")
df79_startweek_long["year"] = pd.to_numeric(df79_startweek_long["year"])
df79_startweek_long["job_num"] = pd.to_numeric(df79_startweek_long["job_num"])
df79_startweek_long = df79_startweek_long.drop(columns="variable")


In [9]:
# First observed employment, before the start/stop merge and any sample exclusions.
entry79 = df79_startweek_long[["CASEID", "Startweek"]].copy()
entry79["Startweek"] = entry79["Startweek"].where(entry79["Startweek"].ge(0))
entry79 = entry79.groupby("CASEID")["Startweek"].min().rename("first_start_week").reset_index()
weeks79 = entry79["first_start_week"]
entry_date79 = pd.Series(pd.NaT, index=entry79.index, dtype="datetime64[ns]")
known79 = weeks79.ge(1)
entry_date79.loc[known79] = pd.Timestamp(1978, 1, 1) + pd.to_timedelta(
    (weeks79.loc[known79] - 1) * 7, unit="D")
entry79["first_entry_year"] = entry_date79.dt.year
entry79["entry_boundary"] = weeks79.le(1).where(weeks79.notna()).astype("Int64")
entry79["entry_date_unknown"] = weeks79.eq(0).astype(int)
entry79["entry_source"] = "NLSY79 recorded job start week"
entry79["entry_observed_only"] = 1
# Week zero means an unknown start before 1978; its entry year stays missing.


In [10]:
# Stop Week
stop_cols = [col for col in df79_exp.columns if col.startswith("STOPWEEK.")]
df79_stopweek_long = df79_exp[["CASEID"] + stop_cols].melt(id_vars="CASEID",value_vars=stop_cols,var_name="variable",value_name="Stopweek")
df79_stopweek_long = df79_stopweek_long.dropna(subset=['Stopweek'])
df79_stopweek_long[["job_num","year"]] = df79_stopweek_long["variable"].str.extract(r"STOPWEEK\.(\d+)_(\d{4})")
df79_stopweek_long["year"] = pd.to_numeric(df79_stopweek_long["year"])
df79_stopweek_long["job_num"] = pd.to_numeric(df79_stopweek_long["job_num"])
df79_stopweek_long = df79_stopweek_long.drop(columns="variable")

# Merging
df79_jobweeks_long = df79_startweek_long.merge(df79_stopweek_long, on=['CASEID', 'year', 'job_num'], how='inner', validate='1:1')
df79_jobweeks_long = df79_jobweeks_long.sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)
df79_jobweeks_long = df79_jobweeks_long.loc[df79_jobweeks_long['Stopweek'].ge(df79_jobweeks_long['Startweek'])].copy()
df79_jobweeks_long['segment'] = df79_jobweeks_long['Stopweek'] - df79_jobweeks_long['Startweek']

# Minus Within Job Gap
df79_gap = pd.read_csv(os.path.join(data_folder, '79-job_gap.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '79-job_gap.cdb'))
df79_gap = df79_gap.rename(columns = rename_dict)
df79_gap = df79_gap.rename(columns = rename_columns)
df79_gap = df79_gap.replace([-1, -2, -3, -4,-5], np.nan)

gap_cols = [c for c in df79_gap.columns if c.startswith(("PERSTAR",'PERSTOP'))]
df79_gap_long = df79_gap[['CASEID']+gap_cols].melt(id_vars = 'CASEID', value_vars = gap_cols, var_name = 'variable')
df79_gap_long = df79_gap_long.dropna(subset=['value'])
df79_gap_long[["gap_var", "job_num", "gap_num", "year"]] = df79_gap_long["variable"].str.extract(r"(PERSTAR|PERSTOP).(\d+)_(\d+)_(\d{4})")
df79_gap_long["job_num"] = pd.to_numeric(df79_gap_long["job_num"])
df79_gap_long["gap_num"] = pd.to_numeric(df79_gap_long["gap_num"])
df79_gap_long["year"] = pd.to_numeric(df79_gap_long["year"])
df79_gap_long = df79_gap_long.drop(columns="variable")
df79_gap_long = df79_gap_long.pivot_table(index=["CASEID", "year", "job_num", "gap_num"],columns="gap_var",values="value",aggfunc="first").reset_index()
df79_gap_long['duration'] = df79_gap_long['PERSTOP'] - df79_gap_long['PERSTAR']
df79_gap_sum = df79_gap_long.groupby(['CASEID','year','job_num'],as_index=False)['duration'].sum()

# Merging with exp df and Generating tenure
df79_exp = df79_jobweeks_long.merge(df79_gap_sum,on=['CASEID','year','job_num'],how='left')
df79_exp['duration'] = df79_exp['duration'].fillna(0)
df79_exp['segment_new'] = df79_exp['segment'] - df79_exp['duration']
# df79_exp = df79_exp.drop(columns='duration')

df79_exp = df79_exp[["CASEID", "job_num", "year", "Startweek", "segment"]].copy()
df79_exp = df79_exp.sort_values(["CASEID", "year", "Startweek", "job_num"])
df79_exp["tenure"] = df79_exp.groupby(["CASEID","job_num"])["segment"].cumsum()
df79_exp = df79_exp[["CASEID", "job_num", "year", "tenure"]].reset_index(drop=True)
df79_exp['tenure'] = df79_exp['tenure']

# Save memory spaces
del df79_startweek_long, df79_stopweek_long, df79_jobweeks_long, df79_gap_long, df79_gap_sum
gc.collect()


## NLSY79 wages and covariates


In [11]:
df79 = pd.read_csv(os.path.join(data_folder, 'nlsy79.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, 'nlsy79.cdb'))
df79 = df79.rename(columns = rename_dict)
df79 = df79.rename(columns=rename_columns)
df79 = df79.replace([-1, -2, -3, -4,-5], np.nan)

# Main job flag before ASVAB exclusions
mainjob_cols = [col for col in df79.columns if col.startswith('CPSJOB.')]
assert mainjob_cols, "Download the EMPLOYERS_ALL_CPSJOB variables and rerun extraction."
df79_mainjob = df79.melt(id_vars=["CASEID"],value_vars=mainjob_cols,var_name="variable",value_name="CPSJOB")
df79_mainjob = df79_mainjob.dropna(subset=['CPSJOB'])
df79_mainjob[["job_num", "year"]] = df79_mainjob["variable"].str.extract(r"CPSJOB\.(\d+)_(\d{4})")
df79_mainjob["job_num"] = df79_mainjob["job_num"].astype(int)
df79_mainjob["year"] = df79_mainjob["year"].astype(int)
df79_mainjob = df79_mainjob.drop(columns=["variable"])

hour_cols = [col for col in df79.columns if col.startswith('HOURSWEEK.')]
wage_cols = [col for col in df79.columns if col.startswith('HRLY_WAGE.')]
occ_cols = [col for col in df79.columns if col.startswith('OCC.')]
asvab_cols = [c for c in df79.columns if c.startswith("ASVAB")]

for c in occ_cols:
    original = df79[c].copy()
    if int(c[-4:]) < 2002:
        df79[c] = original.map(occ70_dict)
    elif int(c[-4:]) == 2002:
        df79[c] = original.map(occ00_dict)
    elif int(c[-4:]) > 2002:
        df79[c] = (original / 10).map(occ00_dict)

for c in wage_cols:
    year = int(c[-4:])
    df79[c] = (df79[c] / 100)/ cpi_map[year]


In [12]:
# Main occupation history before ASVAB and other complete case restrictions.
occ_source79 = df79.melt(id_vars=['CASEID'], value_vars=occ_cols,
                         var_name='variable', value_name='OCC')
occ_source79[['job_num', 'year']] = occ_source79['variable'].str.extract(r'OCC\.(\d+)_(\d{4})').astype(int)
occ_source79 = occ_source79.drop(columns='variable')
# Occupations of the original main employers, before ability or final sample exclusions.
main_occ79 = df79_mainjob.loc[df79_mainjob['CPSJOB'].eq(1)][["CASEID", "year", "job_num"]].merge(
    occ_source79[["CASEID", "year", "job_num", "OCC"]],
    on=["CASEID", "year", "job_num"], how="left", validate="1:1")
main_occ79["occ_main_code"] = main_occ79["OCC"].map(OCC_CODES)
main_count79 = main_occ79.groupby(["CASEID", "year"]).size().rename("main_job_count")

# More than one main employer in a wave leaves that wave's occupation unassigned.
unique_main79 = main_occ79.loc[~main_occ79.duplicated(["CASEID", "year"], keep=False)]
history79 = unique_main79.set_index(["CASEID", "year"])[["job_num", "occ_main_code"]]
history79 = history79.rename(columns={"job_num": "main_job_key"})

# Include missed scheduled waves so lags never skip a missing interview.
all_waves79 = pd.MultiIndex.from_product(
    [sorted(df79["CASEID"].unique()), sorted(survey_year_79)], names=["CASEID", "year"])
history79 = history79.reindex(all_waves79).join(main_count79).reset_index()
history79["main_job_count"] = history79["main_job_count"].fillna(0).astype(int)
for lag in (1, 2):
    history79[f"occ_l{lag}"] = history79.groupby("CASEID")["occ_main_code"].shift(lag)
    history79[f"occ_l{lag}_year"] = history79.groupby("CASEID")["year"].shift(lag)
    history79[f"occ_l{lag}_gap"] = history79["year"] - history79[f"occ_l{lag}_year"]

del occ_source79, main_occ79, unique_main79
gc.collect()


In [13]:
# Standardize ASVAB subtests within birth year and quarter, then apply PCA
df79 = pd.concat([df79,(df79["Q1-3_A~Y_1979"].astype(str)+ "_Q"+ (((df79["Q1-3_A~M_1979"] - 1) // 3) + 1).astype(str)).rename("group_3month")], axis=1)
std_cols = [f"{col}_std" for col in asvab_cols]
df79[std_cols] = df79.groupby("group_3month")[asvab_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=1))

# Applying PCA analysis
valid_r = df79[std_cols].notna().all(axis=1)
valid_df = df79.loc[valid_r, std_cols]
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(valid_df)
print("Variance explained by PC1:", f"{pca.explained_variance_ratio_[0]:.3f}")
df79["ASVAB"] = np.nan
df79.loc[valid_r, "ASVAB"] = pca_scores[:, 0]
df79.drop(columns=asvab_cols+std_cols+["group_3month"], inplace=True)
df79 = df79.dropna(subset=['ASVAB'])


Variance explained by PC1: 0.686


In [14]:
# Hoursweek
df79_hrsweek_long = df79.melt(id_vars=["CASEID"],value_vars=hour_cols,var_name="variable",value_name="HOURSWEEK")
df79_hrsweek_long = df79_hrsweek_long.dropna(subset=['HOURSWEEK'])
df79_hrsweek_long[["job_num", "year"]] = df79_hrsweek_long["variable"].str.extract(r"HOURSWEEK\.(\d+)_(\d{4})")
df79_hrsweek_long["job_num"] = df79_hrsweek_long["job_num"].astype(int)
df79_hrsweek_long["year"] = df79_hrsweek_long["year"].astype(int)
df79_hrsweek_long = df79_hrsweek_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Wage
df79_wage_long = df79.melt(id_vars=["CASEID"],value_vars=wage_cols,var_name="variable",value_name="WAGE")
df79_wage_long = df79_wage_long.dropna(subset=['WAGE'])
df79_wage_long[["job_num", "year"]] = df79_wage_long["variable"].str.extract(r"HRLY_WAGE\.(\d+)_(\d{4})")
df79_wage_long["job_num"] = df79_wage_long["job_num"].astype(int)
df79_wage_long["year"] = df79_wage_long["year"].astype(int)
df79_wage_long = df79_wage_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Occupation
df79_occ_long = df79.melt(id_vars=["CASEID"],value_vars=occ_cols,var_name="variable",value_name="OCC")
df79_occ_long = df79_occ_long.dropna(subset=['OCC'])
df79_occ_long[["job_num", "year"]] = df79_occ_long["variable"].str.extract(r"OCC\.(\d+)_(\d{4})")
df79_occ_long["job_num"] = df79_occ_long["job_num"].astype(int)
df79_occ_long["year"] = df79_occ_long["year"].astype(int)
df79_occ_long = df79_occ_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Education
educ_cols = [c for c in df79.columns if c.startswith("HGCREV")]
educ_long = df79[["CASEID"] + educ_cols].melt(id_vars="CASEID",var_name="variable",value_name="HGCREV")
educ_long['year'] = educ_long['variable'].str.extract(r"(\d{4})$")[0].astype(int)
educ_long = educ_long.drop(columns='variable').sort_values(['CASEID','year'])
educ_long['HGCREV'] = educ_long.groupby("CASEID")['HGCREV'].ffill()

# Sample Weight
weight_cols = [c for c in df79.columns if c.startswith("SAMPWEIGHT_")]
weight_long = df79[["CASEID"] + weight_cols].melt(id_vars="CASEID",var_name="variable",value_name="SampleWeight")
weight_long['year'] = weight_long['variable'].str.extract(r"(\d{4})$")[0].astype(int)
weight_long["SampleWeight"] = weight_long["SampleWeight"] / 100
weight_long = weight_long.drop(columns='variable')

# Age, Sex, Race, ASVAB
person_vars = df79[['CASEID', 'sex', 'race', 'ASVAB', 'Q1-3_A~Y_1979']].drop_duplicates(subset=['CASEID']).copy()
person_vars["Q1-3_A~Y_1979"] = person_vars['Q1-3_A~Y_1979'] + 1900
person_vars['race'] = person_vars['race'].map(race_mapping)
person_vars['sex'] = person_vars['sex'].map(sex_mapping)

# Merge
df79_long = reduce(lambda left, right: pd.merge(left,right,on=["CASEID", "year", "job_num"],how="left"),[df79_exp,df79_hrsweek_long,df79_wage_long,df79_occ_long])
job_start79 = df79_exp.groupby(["CASEID", "job_num"], as_index=False)["year"].min().rename(columns={"year": "start_year"})
job_educ79 = job_start79.merge(educ_long[["CASEID", "year", "HGCREV"]].rename(columns={"year": "start_year", "HGCREV": "Educ"}),on=["CASEID", "start_year"],how="left",validate="m:1")
df79_long = df79_long.merge(job_educ79,on=["CASEID", "job_num"],how='left',validate='m:1')
df79_long = df79_long.merge(weight_long,on=["CASEID", "year"],how='left',validate='m:1')
df79_long = df79_long.merge(person_vars,on='CASEID',how='left',validate='m:1')
df79_long['Age'] = df79_long['year'] - df79_long['Q1-3_A~Y_1979']
df79_long = df79_long.drop(columns=['Q1-3_A~Y_1979','start_year'])

# Generating experience from tenure
last = df79_long.sort_values("year").groupby(["CASEID", "job_num"])["tenure"].last()
prior_jobs = last.groupby(level=0).cumsum() - last
df79_long["experience"] = df79_long["tenure"] + pd.MultiIndex.from_frame(df79_long[["CASEID", "job_num"]]).map(prior_jobs)
df79_long['experience'] = df79_long['experience'] / 52
df79_long['tenure'] = df79_long['tenure'] / 52
# Start of final sample accounting: the merged employment panel before final restrictions.
sample_rows79 = [{
    "Restriction": "Merged employment panel",
    "Rows removed": 0,
    "Rows remaining": len(df79_long),
    "Respondents remaining": df79_long["CASEID"].nunique()
}]

# Keep the CPS main employer after constructing employment history.
df79_long = df79_long.merge(df79_mainjob, on=["CASEID", "year", "job_num"],
                            how="left", validate="1:1")
rows_before = len(df79_long)
df79_long = df79_long.loc[df79_long["CPSJOB"].eq(1)].copy()
df79_long = df79_long.drop(columns="CPSJOB")
sample_rows79.append({
    "Restriction": "CPS main employer",
    "Rows removed": rows_before - len(df79_long),
    "Rows remaining": len(df79_long),
    "Respondents remaining": df79_long["CASEID"].nunique()
})

columns79 = ["SampleWeight", "ASVAB", "sex", "Age", "race", "HOURSWEEK",
             "WAGE", "OCC", "Educ", "tenure", "experience"]
# Sequential deletions for the appendix. Rows can fail more than one condition.
# Each row below counts only the additional loss after the preceding restrictions.
columns_to_check79 = list(dict.fromkeys(columns79 + list(df79_long.columns)))
for column in columns_to_check79:
    rows_before = len(df79_long)
    df79_long = df79_long.dropna(subset=[column])
    sample_rows79.append({
        "Restriction": "Nonmissing " + column,
        "Rows removed": rows_before - len(df79_long),
        "Rows remaining": len(df79_long),
        "Respondents remaining": df79_long["CASEID"].nunique()
    })
df79_long = df79_long.reset_index(drop=True)
sample_table79 = pd.DataFrame(sample_rows79)


### NLSY79 sample construction table
Counts begin with the merged employment panel and show sequential deletions.
They precede the survey year and employer number restrictions applied in Stata.


In [15]:
display(sample_table79)


,Restriction,Rows removed,Rows remaining,Respondents remaining
0,Merged employment panel,0,340320,12425
1,CPS main employer,115695,224625,12264
2,Nonmissing SampleWeight,9447,215178,11527
3,Nonmissing ASVAB,0,215178,11527
4,Nonmissing sex,0,215178,11527
5,Nonmissing Age,0,215178,11527
6,Nonmissing race,1705,213473,11425
7,Nonmissing HOURSWEEK,1235,212238,11422
8,Nonmissing WAGE,10808,201430,11393
9,Nonmissing OCC,5342,196088,11381


### NLSY79 instruments
Elapsed time is measured from the first observed employment year. Occupation histories use
scheduled survey waves; a missed wave remains missing. These variables are added after
cleaning so missing instruments do not remove employment rows.


In [16]:
# Add instruments after complete case selection; missing instruments do not delete rows.
df79_long = df79_long.merge(entry79, on="CASEID", how="left", validate="m:1")
df79_long = df79_long.merge(history79, on=["CASEID", "year"], how="left", validate="m:1")
df79_long["potexp"] = df79_long["year"] - df79_long["first_entry_year"]
df79_long["entry_after_observation"] = df79_long["potexp"].lt(0).astype(int)
df79_long["potexp"] = df79_long["potexp"].where(df79_long["potexp"].ge(0))
df79_long["potexp_2"] = df79_long["potexp"] ** 2
df79_long["potexp_3"] = df79_long["potexp"] ** 3
df79_long["educ_potexp"] = df79_long["Educ"] * df79_long["potexp"]
df79_long["asvab_potexp"] = df79_long["ASVAB"] * df79_long["potexp"]

# Occupation indicators use the previous two scheduled waves of the main employer.
df79_long["occ_code"] = df79_long["OCC"].map(OCC_CODES)
df79_long["is_main_job"] = df79_long["job_num"].eq(df79_long["main_job_key"]).astype(int)
ready79 = (df79_long["is_main_job"].eq(1) & df79_long["occ_main_code"].notna()
           & df79_long["occ_l1"].notna() & df79_long["occ_l2"].notna())
df79_long["occ_iv_ready"] = ready79.astype(int)
df79_long["occ_iv_annual"] = (
    ready79 & df79_long["occ_l1_gap"].eq(1) & df79_long["occ_l2_gap"].eq(2)).astype(int)
for j in OCC_CODES.values():
    for k in OCC_CODES.values():
        df79_long[f"zocc_{j}{k}"] = (
            (df79_long["occ_l1"].eq(j) & df79_long["occ_l2"].eq(k))
            .astype(float).where(ready79))


### Export NLSY79


In [17]:
df79_long.to_csv(CLEANED_DIR / 'nlsy79_cleaned.csv', index=False)


## NLSY97 column names and employer tenure


In [18]:
def rename_columns_97(col):
    if 'PUBID' in col:
        return 'PUBID'
    elif 'YEMP' in col:
        clean_name = col.replace('YEMP_', '')
        if '~' in clean_name:
            clean_name = clean_name.replace('~','_')
        elif 'OCCODE-2002' in clean_name:
            clean_name = clean_name.replace('OCCODE-2002','Occupation')
        return clean_name
    elif 'EMP_' in col:
        clean_name = col.replace('EMP_','')
        if '_WEEK_' in clean_name:
            clean_name = clean_name.replace('WEEK_','')
        if 'GAP' in clean_name:
            match_gap = re.match(r'GAP_(.+)_(\d{4})\.(\d+)\.(\d+)$',clean_name)
            if match_gap:
                var, year, job_num, gap_num = match_gap.groups()
                return f"GAP_{var}.{gap_num}_{year}.{job_num}"
        return clean_name
    elif 'CV_' in col:
        clean_name = col.replace('CV_','')
        if 'WKSWK_JOB_DLI' in clean_name:
            clean_name = clean_name.replace('WKSWK_JOB_DLI','Tenure')
        elif 'MAINJOB_FLG' in clean_name:
            clean_name = clean_name.replace('MAINJOB_FLG','MainJob')
        elif 'HRLY_COMPENSATION' in clean_name:
            clean_name = clean_name.replace('HRLY_COMPENSATION','Wage')
        elif 'HRS_PER_WEEK' in clean_name:
            clean_name = clean_name.replace('HRS_PER_WEEK','Hours')

        return clean_name
    elif 'YSCH-' in col:
        if '5000' in col:
            return 'Educ_1997'
        elif '3112' in col:
            return f'Educ_{col[-4:]}'
    elif 'SAMPLING_' in col:
        return f'SampleWeight_{col[-4:]}'
    elif 'KEY!' in col:
        clean_name = col.replace('KEY!','')
        return clean_name
    return col


In [19]:
df97_exp = pd.read_csv(os.path.join(data_folder, '97-tenure.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '97-tenure.cdb'))
df97_exp = df97_exp.rename(columns = rename_dict)
df97_exp = df97_exp.rename(columns = rename_columns_97)
df97_exp = df97_exp.replace([-1, -2, -3, -4,-5,], np.nan)
mainjob_records = []
for index, row in df97_exp.iterrows():
    pubid = row["PUBID"]
    for year in survey_year_97:
        mainjob_col = f"MainJob_{year}"
        if mainjob_col not in df97_exp.columns:
            continue
        mainjob_value = row[mainjob_col]
        if pd.isna(mainjob_value):
            continue
        suffix = f"{int(mainjob_value):02d}"
        uid_col = f"UID.{suffix}_{year}"
        tenure_col = f"Tenure.{suffix}_{year}"
        if uid_col not in df97_exp.columns:
            continue
        if tenure_col not in df97_exp.columns:
            continue
        uid = row[uid_col]
        tenure = row[tenure_col]
        if pd.isna(uid):
            continue
        mainjob_records.append({
            "PUBID": pubid,
            "year": int(year),
            "uid": int(uid),
            "tenure": tenure,
            "suffix": suffix
        })
df97_mainjob = pd.DataFrame(mainjob_records).sort_values(["PUBID", "year"]).reset_index(drop=True)


In [20]:
exp_row_lookup = {}
for index, row in df97_exp.iterrows():
    if pd.notna(row["PUBID"]):
        exp_row_lookup[int(row["PUBID"])] = row

mainjob_original = df97_mainjob.sort_values(["PUBID", "year"]).reset_index(drop=True).copy()
additional_rows = []
previous_mainjob = {}

for index, current in mainjob_original.iterrows():
    pubid = int(current["PUBID"])
    if pubid not in previous_mainjob:
        previous_mainjob[pubid] = current.to_dict()
        continue
    previous = previous_mainjob[pubid]
    previous_year = int(previous["year"])
    current_year = int(current["year"])
    previous_uid = int(previous["uid"])
    current_uid = int(current["uid"])

    if previous_uid != current_uid:
        exp_row = exp_row_lookup[pubid]
        # Look for the old employer in the current wave and the new employer in the prior wave.
        for target_uid, target_year in [(previous_uid, current_year), (current_uid, previous_year)]:
            for column in df97_exp.columns:
                if not (column.startswith("UID.") and column.endswith(f"_{target_year}")):
                    continue
                uid_value = exp_row[column]
                if pd.notna(uid_value) and int(uid_value) == target_uid:
                    suffix = column.replace("UID.", "").split("_")[0]
                    suffix = f"{int(suffix):02d}"
                    tenure = exp_row.get(f"Tenure.{suffix}_{target_year}", np.nan)
                    if pd.notna(tenure):
                        additional_rows.append({"PUBID": pubid, "year": target_year,
                                                "uid": target_uid, "tenure": tenure,
                                                "suffix": suffix})
                    break
    previous_mainjob[pubid] = current.to_dict()

additional_jobs = pd.DataFrame(additional_rows,columns=["PUBID","year","uid","tenure","suffix"])
df97_mainjob = pd.concat([mainjob_original, additional_jobs],ignore_index=True)
df97_mainjob = df97_mainjob.drop_duplicates(subset=['PUBID', 'year', 'uid']).copy().sort_values(['PUBID', 'year', 'uid']).reset_index(drop=True)
df97_mainjob = df97_mainjob.sort_values(["PUBID", "year"], kind="stable").reset_index(drop=True)
df97_mainjob["jobnum"] = df97_mainjob.groupby("PUBID")["uid"].transform(lambda x: x.ne(x.shift()).cumsum()).astype("Int64")


## NLSY97 wages, covariates, and employment histories


In [21]:
df97 = pd.read_csv(os.path.join(data_folder, 'nlsy97.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, 'nlsy97.cdb'))
df97 = df97.rename(columns = rename_dict)
df97 = df97.rename(columns = rename_columns_97)
df97 = df97.replace([-1, -2, -3, -4,-5], np.nan)
df97.drop(columns=['SAMPLE_TYPE_1997'],inplace=True)

asvab_cols = [c for c in df97.columns if c.startswith("ASVAB")]
subtests = ["GS", "AR", "WK", "PC", "NO", "CS", "MK", "MC"]
for test in subtests:
    pos = f"ASVAB_{test}_ABILITY_EST_POS"
    neg = f"ASVAB_{test}_ABILITY_EST_NEG"
    df97[f"ASVAB_{test}"] = (df97[pos]/1000).combine_first(-df97[neg]/1000)
    df97 = df97.drop(columns=[pos,neg])
asvab_cols = [c for c in df97.columns if c.startswith("ASVAB")]
col97 = [c[:-5] for c in df97.columns if c != 'PUBID']
df97["group_3month"] = (df97["BDATE_Y_1997"].astype(str) + "_Q" + (((df97["BDATE_M_1997"] - 1) // 3) + 1).astype(str))

wage_cols = [c for c in df97.columns if c.startswith('Wage')]
occ_cols = [c for c in df97.columns if c.startswith('Occupation')]

for c in wage_cols:
    year = int(c[-4:])
    df97[c] = (df97[c] / 100)/ cpi_map[year]

for c in occ_cols:
    original = df97[c].copy()
    if int(c[-4:]) < 2002:
        df97[c] = original.map(occ70_dict)
    elif int(c[-4:]) == 2002:
        df97[c] = original.map(occ00_dict)
    elif int(c[-4:]) > 2002:
        df97[c] = (original / 10).map(occ00_dict)


In [22]:
# The original main employer, not the additional employer rows, defines occupation.
occ_source97 = df97.melt(id_vars=['PUBID'], value_vars=occ_cols,
                         var_name='variable', value_name='Occupation')
occ_source97[['suffix', 'year']] = occ_source97['variable'].str.extract(r'Occupation\.(\d+)_(\d{4})')
occ_source97['suffix'] = occ_source97['suffix'].astype(int).map(lambda x: f'{x:02d}')
occ_source97['year'] = occ_source97['year'].astype(int)
occ_source97 = occ_source97.drop(columns='variable')
# Occupations of the original main employers, before ability or final sample exclusions.
main_occ97 = mainjob_original[["PUBID", "year", "suffix"]].merge(
    occ_source97[["PUBID", "year", "suffix", "Occupation"]],
    on=["PUBID", "year", "suffix"], how="left", validate="1:1")
main_occ97["occ_main_code"] = main_occ97["Occupation"].map(OCC_CODES)
main_count97 = main_occ97.groupby(["PUBID", "year"]).size().rename("main_job_count")

# More than one main employer in a wave leaves that wave's occupation unassigned.
unique_main97 = main_occ97.loc[~main_occ97.duplicated(["PUBID", "year"], keep=False)]
history97 = unique_main97.set_index(["PUBID", "year"])[["suffix", "occ_main_code"]]
history97 = history97.rename(columns={"suffix": "main_job_key"})

# Include missed scheduled waves so lags never skip a missing interview.
all_waves97 = pd.MultiIndex.from_product(
    [sorted(df97["PUBID"].unique()), sorted(survey_year_97)], names=["PUBID", "year"])
history97 = history97.reindex(all_waves97).join(main_count97).reset_index()
history97["main_job_count"] = history97["main_job_count"].fillna(0).astype(int)
for lag in (1, 2):
    history97[f"occ_l{lag}"] = history97.groupby("PUBID")["occ_main_code"].shift(lag)
    history97[f"occ_l{lag}_year"] = history97.groupby("PUBID")["year"].shift(lag)
    history97[f"occ_l{lag}_gap"] = history97["year"] - history97[f"occ_l{lag}_year"]

# Match the final employer UID rather than its survey roster suffix.
main_uid97 = mainjob_original[["PUBID", "year", "uid"]].rename(columns={"uid": "main_job_key"})
history97 = history97.drop(columns="main_job_key").merge(
    main_uid97, on=["PUBID", "year"], how="left", validate="1:1")

del occ_source97, main_occ97, unique_main97
gc.collect()


In [23]:
# First observed employment from weekly employer status.
weekly_mapping = create_rnum_dict(data_folder / '97-emp_weekly_array.cdb')
weekly_columns = {raw: name for raw, name in weekly_mapping.items()
                  if name.startswith('EMP_STATUS_') or name.startswith('PUBID')}
weekly97 = pd.read_csv(data_folder / '97-emp_weekly_array.csv', usecols=list(weekly_columns))
weekly97 = weekly97.rename(columns=weekly_columns)
weekly_id = next(c for c in weekly97 if c.startswith('PUBID'))
status_columns = sorted(
    (c for c in weekly97 if c.startswith('EMP_STATUS_')),
    key=lambda c: tuple(map(int, re.search(r'(\d{4})\.(\d+)$', c).groups())))
status97 = weekly97[status_columns].to_numpy()
employed97 = status97 >= 9700
has_work97 = employed97.any(axis=1)
first_work97 = employed97.argmax(axis=1)
status_years97 = np.array([int(re.search(r'(\d{4})\.', c).group(1)) for c in status_columns])
status_weeks97 = np.array([int(c.rsplit('.', 1)[1]) for c in status_columns])
definite97 = np.isin(status97, [1, 2, 4, 5, 6]) | employed97
first_known97 = definite97.argmax(axis=1)
entry97 = pd.DataFrame({
    'PUBID': weekly97[weekly_id],
    'first_entry_year': np.where(has_work97, status_years97[first_work97], np.nan),
    'first_start_week': np.where(has_work97, status_weeks97[first_work97], np.nan),
    'entry_boundary': pd.array(
        np.where(has_work97, first_work97 == first_known97, np.nan), dtype='Int64'),
    'entry_date_unknown': (~has_work97).astype(int),
    'entry_source': 'NLSY97 weekly employer status',
    'entry_observed_only': 1
})
# Status codes 0 through 6 are categories; employer identifiers start at 9700.
del weekly97, status97, employed97, definite97
gc.collect()


In [24]:
# Standardize ASVAB subtests within birth year and quarter, then apply PCA
std_cols = [f"{col}_std" for col in asvab_cols]
df97 = df97.copy()
df97[std_cols] = df97.groupby("group_3month")[asvab_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=1))

# Applying PCA analysis
valid_r = df97[std_cols].notna().all(axis=1)
valid_df = df97.loc[valid_r, std_cols]
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(valid_df)
print("Variance explained by PC1:", f"{pca.explained_variance_ratio_[0]:.3f}")
df97["ASVAB"] = np.nan
df97.loc[valid_r, "ASVAB"] = pca_scores[:, 0]
df97.drop(columns=asvab_cols+std_cols+["group_3month","BDATE_M_1997"], inplace=True)
df97 = df97.dropna(subset=['ASVAB'])


Variance explained by PC1: 0.672


In [25]:
# Hoursweek
hour_cols = [c for c in df97.columns if c.startswith('Hours')]
df97_hrs = df97.melt(id_vars=["PUBID"],value_vars=hour_cols,var_name="variable",value_name="Hours")
df97_hrs = df97_hrs.dropna(subset=['Hours'])
df97_hrs[['suffix','year']] = df97_hrs['variable'].str.extract(r"Hours\.(\d+)_(\d{4})")
df97_hrs['suffix'] = df97_hrs['suffix'].astype(int)
df97_hrs['year'] = df97_hrs['year'].astype(int)
df97_hrs['suffix'] = df97_hrs['suffix'].apply(lambda x:f'{int(x):02d}')
df97_hrs = df97_hrs.drop(columns=['variable']).sort_values(['PUBID','year','suffix']).reset_index(drop=True)

# Wage
df97_wage = df97.melt(id_vars=["PUBID"],value_vars=wage_cols,var_name="variable",value_name="Wage")
df97_wage = df97_wage.dropna(subset=['Wage'])
df97_wage[['suffix','year']] = df97_wage['variable'].str.extract(r"Wage\.(\d+)_(\d{4})")
df97_wage['suffix'] = df97_wage['suffix'].astype(int)
df97_wage['year'] = df97_wage['year'].astype(int)
df97_wage['suffix'] = df97_wage['suffix'].apply(lambda x:f'{int(x):02d}')
df97_wage = df97_wage.drop(columns=['variable']).sort_values(['PUBID','year','suffix']).reset_index(drop=True)

# Occupation
df97_occ = df97.melt(id_vars=["PUBID"],value_vars=occ_cols,var_name="variable",value_name="Occupation")
df97_occ = df97_occ.dropna(subset=['Occupation'])
df97_occ[['suffix','year']] = df97_occ['variable'].str.extract(r"Occupation\.(\d+)_(\d{4})")
df97_occ['suffix'] = df97_occ['suffix'].astype(int)
df97_occ['year'] = df97_occ['year'].astype(int)
df97_occ['suffix'] = df97_occ['suffix'].apply(lambda x:f'{int(x):02d}')
df97_occ = df97_occ.drop(columns=['variable']).sort_values(['PUBID','year','suffix']).reset_index(drop=True)

# Education 
educ_cols = [c for c in df97.columns if c.startswith('Educ')]
df97_educ = df97[['PUBID']+educ_cols].melt(id_vars='PUBID',var_name='variable',value_name='Education')
df97_educ['year'] = df97_educ['variable'].str.extract(r"(\d{4})$")[0].astype(int)
df97_educ = df97_educ.drop(columns='variable').sort_values(['PUBID','year'])
df97_educ["Education"] = df97_educ["Education"].replace(95, np.nan)
df97_educ['Education'] = df97_educ.groupby("PUBID")['Education'].ffill()

# Sample Weight 
weight_cols = [c for c in df97.columns if c.startswith('Sample')]
df97_weight = df97[['PUBID']+weight_cols].melt(id_vars='PUBID',var_name='variable',value_name='Sample Weight')
df97_weight['year'] = df97_weight['variable'].str.extract(r"(\d{4})$")[0].astype(int)
df97_weight['Sample Weight'] = df97_weight['Sample Weight'] / 100
df97_weight = df97_weight.loc[df97_weight['Sample Weight'].ne(0)].copy()
df97_weight = df97_weight.drop(columns='variable').reset_index(drop=True)

# Age, Sex, Race, ASVAB
df97_person = df97[['PUBID', 'SEX_1997', 'RACE_ETHNICITY_1997', 'ASVAB', 'BDATE_Y_1997']].drop_duplicates(subset=['PUBID']).copy()
# df97_person['RACE_ETHNICITY_1997'] = df97_person['RACE_ETHNICITY_1997'].map(race_mapping)
df97_person['SEX_1997'] = df97_person['SEX_1997'].map(sex_mapping)
df97_person['RACE_ETHNICITY_1997'] = df97_person['RACE_ETHNICITY_1997'].map(race_map_97)
df97_person = df97_person.rename(columns={'SEX_1997':'Sex', 'RACE_ETHNICITY_1997':'Race'})

# Merge
df97_long = reduce(lambda left, right: pd.merge(left,right,on=["PUBID", "year", "suffix"],how="left"),[df97_mainjob,df97_hrs,df97_wage,df97_occ])
job_start = df97_mainjob.groupby(["PUBID", "jobnum"], as_index=False)["year"].min().rename(columns={"year": "start_year"})
job_educ = job_start.merge(df97_educ[["PUBID", "year", "Education"]].rename(columns={"year": "start_year", "Education": "Educ"}),on=["PUBID", "start_year"],how="left",validate="m:1")
df97_long = df97_long.merge(job_educ[["PUBID", "jobnum", "Educ"]],on=["PUBID", "jobnum"],how="left",validate="m:1")
df97_long = df97_long.merge(df97_weight,on=["PUBID", "year"],how='left',validate='m:1')
df97_long = df97_long.merge(df97_person,on='PUBID',how='left',validate='m:1')
df97_long['Age'] = df97_long['year'] - df97_long['BDATE_Y_1997']
df97_long.drop(columns=['suffix','BDATE_Y_1997'],inplace=True)

# Generate Experience
last = (df97_long.sort_values(["PUBID", "year"]).groupby(["PUBID", "jobnum"])["tenure"].last())
prior_jobs = last.groupby(level=0).cumsum() - last
df97_long["Experience"] = (df97_long["tenure"] + pd.MultiIndex.from_frame(df97_long[["PUBID", "jobnum"]]).map(prior_jobs)) / 52
df97_long['tenure'] = df97_long['tenure'] / 52
# Start of final sample accounting after constructing the employment panel.
sample_rows97 = [{
    "Restriction": "Merged employment panel",
    "Rows removed": 0,
    "Rows remaining": len(df97_long),
    "Respondents remaining": df97_long["PUBID"].nunique()
}]
columns97 = ["Sample Weight", "ASVAB", "Sex", "Age", "Race", "Hours",
             "Wage", "Occupation", "Educ", "tenure", "Experience"]
# Sequential deletions for the appendix. Rows can fail more than one condition.
# Each row below counts only the additional loss after the preceding restrictions.
columns_to_check97 = list(dict.fromkeys(columns97 + list(df97_long.columns)))
for column in columns_to_check97:
    rows_before = len(df97_long)
    df97_long = df97_long.dropna(subset=[column])
    sample_rows97.append({
        "Restriction": "Nonmissing " + column,
        "Rows removed": rows_before - len(df97_long),
        "Rows remaining": len(df97_long),
        "Respondents remaining": df97_long["PUBID"].nunique()
    })
df97_long = df97_long.reset_index(drop=True)
sample_table97 = pd.DataFrame(sample_rows97)


### NLSY97 sample construction table
Counts begin with the merged employment panel and show sequential deletions.
They precede the employer number restrictions applied in Stata.


In [26]:
display(sample_table97)


,Restriction,Rows removed,Rows remaining,Respondents remaining
0,Merged employment panel,0,161638,8862
1,Nonmissing Sample Weight,55429,106209,6614
2,Nonmissing ASVAB,0,106209,6614
3,Nonmissing Sex,0,106209,6614
4,Nonmissing Age,0,106209,6614
5,Nonmissing Race,0,106209,6614
6,Nonmissing Hours,751,105458,6612
7,Nonmissing Wage,5407,100051,6592
8,Nonmissing Occupation,32023,68028,5385
9,Nonmissing Educ,0,68028,5385


### NLSY97 instruments


In [27]:
# Add instruments after complete case selection; missing instruments do not delete rows.
df97_long = df97_long.merge(entry97, on="PUBID", how="left", validate="m:1")
df97_long = df97_long.merge(history97, on=["PUBID", "year"], how="left", validate="m:1")
df97_long["potexp"] = df97_long["year"] - df97_long["first_entry_year"]
df97_long["entry_after_observation"] = df97_long["potexp"].lt(0).astype(int)
df97_long["potexp"] = df97_long["potexp"].where(df97_long["potexp"].ge(0))
df97_long["potexp_2"] = df97_long["potexp"] ** 2
df97_long["potexp_3"] = df97_long["potexp"] ** 3
df97_long["educ_potexp"] = df97_long["Educ"] * df97_long["potexp"]
df97_long["asvab_potexp"] = df97_long["ASVAB"] * df97_long["potexp"]

# Occupation indicators use the previous two scheduled waves of the main employer.
df97_long["occ_code"] = df97_long["Occupation"].map(OCC_CODES)
df97_long["is_main_job"] = df97_long["uid"].eq(df97_long["main_job_key"]).astype(int)
ready97 = (df97_long["is_main_job"].eq(1) & df97_long["occ_main_code"].notna()
           & df97_long["occ_l1"].notna() & df97_long["occ_l2"].notna())
df97_long["occ_iv_ready"] = ready97.astype(int)
df97_long["occ_iv_annual"] = (
    ready97 & df97_long["occ_l1_gap"].eq(1) & df97_long["occ_l2_gap"].eq(2)).astype(int)
for j in OCC_CODES.values():
    for k in OCC_CODES.values():
        df97_long[f"zocc_{j}{k}"] = (
            (df97_long["occ_l1"].eq(j) & df97_long["occ_l2"].eq(k))
            .astype(float).where(ready97))


### Export NLSY97


In [28]:
df97_long.to_csv(CLEANED_DIR / 'nlsy97_cleaned.csv', index=False)


## Combine the cohorts


In [29]:
df_79 = df79_long.copy()
df_97 = df97_long.copy()
df_79["cohort"] = 0
df_97["cohort"] = 1
rename_97 = {
    "PUBID": "ID", "year":"Year", 'uid':"Uid", 'tenure':"Tenure",'jobnum':'job_num', 'Hours':'Hour', 'Wage':"Wage", 'Occupation':'Occ','Educ':'Educ',
    "Sample Weight": "SampleWeight","Sex": "Sex", 'Race':'Race', 'ASVAB':"ASVAB", 'Age':"Age",'Experience':'Exp'
}
rename_79 = {
    'CASEID':"ID", 'job_num':'job_num', 'year':'Year', 'tenure':'Tenure','HOURSWEEK':'Hour','WAGE':'Wage','OCC':'Occ','Educ':'Educ',
    'SampleWeight':'SampleWeight', 'sex':'Sex','race':'Race','ASVAB':'ASVAB','Age':'Age','experience':'Exp'
}
df_79.rename(columns=rename_79,inplace=True)
df_97.rename(columns=rename_97,inplace=True)
df_main = pd.concat([df_79, df_97],axis=0,ignore_index=True,sort=False)
df_main = df_main.sort_values(["cohort", "ID", "Year", "job_num"]).reset_index(drop=True)
assert len(df_main) == len(df79_long) + len(df97_long)


### Export combined data


In [30]:
df_main.to_csv(CLEANED_DIR / 'nlsy_merged.csv', index=False)
